# Лабораторная работа №1

## Apache Spark. Анализ данных велопарковок Сан-Франциско


## Подготовка окружения и чтение данных


In [1]:
from pathlib import Path
from datetime import datetime
from math import radians, sin, cos, sqrt, atan2

from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession

conf = SparkConf().setAppName("Lab1_PySpark").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)
spark = SparkSession.builder.config(conf=conf).getOrCreate()

trips_path = str("trips.csv")
stations_path = str("stations.csv")

print(f"Spark version: {sc.version}")
print(f"Trips path: {trips_path}")
print(f"Stations path: {stations_path}")


Spark version: 4.0.2
Trips path: trips.csv
Stations path: stations.csv


In [2]:
trip_data = sc.textFile(trips_path)
trip_header = trip_data.first()
trip_rows = trip_data.filter(lambda row: row != trip_header).map(lambda row: row.split(",", -1))

station_data = sc.textFile(stations_path)
station_header = station_data.first()
station_rows = station_data.filter(lambda row: row != station_header).map(lambda row: row.split(",", -1))

print(trip_header)
print(station_header)


id,duration,start_date,start_station_name,start_station_id,end_date,end_station_name,end_station_id,bike_id,subscription_type,zip_code
id,name,lat,long,dock_count,city,installation_date


In [3]:
def parse_station(row):
    try:
        return {
            "station_id": int(row[0]),
            "name": row[1],
            "lat": float(row[2]),
            "long": float(row[3]),
            "dock_count": int(row[4]),
            "city": row[5],
            "installation_date": datetime.strptime(row[6], "%m/%d/%Y")
        }
    except Exception:
        return None


def parse_trip(row):
    try:
        if not row[1] or not row[2] or not row[5] or not row[8]:
            return None
        return {
            "trip_id": int(row[0]),
            "duration": int(row[1]),
            "start_date": datetime.strptime(row[2], "%m/%d/%Y %H:%M"),
            "start_station_name": row[3],
            "start_station_id": int(row[4]),
            "end_date": datetime.strptime(row[5], "%m/%d/%Y %H:%M"),
            "end_station_name": row[6],
            "end_station_id": int(row[7]),
            "bike_id": int(row[8]),
            "subscription_type": row[9],
            "zip_code": row[10].strip()
        }
    except Exception:
        return None


stations = station_rows.map(parse_station).filter(lambda row: row is not None).cache()
trips = trip_rows.map(parse_trip).filter(lambda row: row is not None).cache()

print(f"Stations: {stations.count()}")
print(f"Trips: {trips.count()}")
print(stations.take(1)[0])
print(trips.take(1)[0])


Stations: 70
Trips: 396085
{'station_id': 2, 'name': 'San Jose Diridon Caltrain Station', 'lat': 37.329732, 'long': -121.90178200000001, 'dock_count': 27, 'city': 'San Jose', 'installation_date': datetime.datetime(2013, 8, 6, 0, 0)}
{'trip_id': 4130, 'duration': 71, 'start_date': datetime.datetime(2013, 8, 29, 10, 16), 'start_station_name': 'Mountain View City Hall', 'start_station_id': 27, 'end_date': datetime.datetime(2013, 8, 29, 10, 17), 'end_station_name': 'Mountain View City Hall', 'end_station_id': 27, 'bike_id': 48, 'subscription_type': 'Subscriber', 'zip_code': '97214'}


## Выполнение заданий


### Задание 1. Найти велосипед с максимальным временем пробега


In [4]:
bike_id, total_duration = (
    trips
    .map(lambda trip: (trip["bike_id"], trip["duration"]))
    .reduceByKey(lambda left, right: left + right)
    .takeOrdered(1, key=lambda item: -item[1])[0]
)

bike_trip_count = trips.filter(lambda trip: trip["bike_id"] == bike_id).count()

print(f"Bike ID: {bike_id}")
print(f"Total duration: {total_duration} sec ({total_duration / 3600:.2f} hours)")
print(f"Trips count: {bike_trip_count}")


Bike ID: 466
Total duration: 3421296 sec (950.36 hours)
Trips count: 846


### Задание 2. Найти наибольшее геодезическое расстояние между станциями


In [5]:
def haversine(lat1, lon1, lat2, lon2):
    radius_km = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return radius_km * c

station_points = stations.map(lambda station: (
    station["station_id"],
    station["name"],
    station["lat"],
    station["long"]
))

max_distance_pair = (
    station_points
    .cartesian(station_points)
    .filter(lambda pair: pair[0][0] < pair[1][0])
    .map(lambda pair: (
        pair[0][0],
        pair[0][1],
        pair[1][0],
        pair[1][1],
        haversine(pair[0][2], pair[0][3], pair[1][2], pair[1][3])
    ))
    .takeOrdered(1, key=lambda item: -item[4])[0]
)

print(
    f"Max distance: {max_distance_pair[4]:.3f} km between "
    f"'{max_distance_pair[1]}' and '{max_distance_pair[3]}'"
)


Max distance: 69.921 km between 'SJSU - San Salvador at 9th' and 'Embarcadero at Sansome'


### Задание 3. Найти путь велосипеда с максимальным временем пробега через станции


In [6]:
max_bike_path = (
    trips
    .filter(lambda trip: trip["bike_id"] == bike_id)
    .sortBy(lambda trip: trip["start_date"])
    .cache()
)

path_segments = max_bike_path.map(lambda trip: (
    trip["start_date"].strftime("%Y-%m-%d %H:%M"),
    trip["start_station_name"],
    trip["end_station_name"],
    trip["duration"]
)).collect()

print(f"Bike {bike_id} path length: {len(path_segments)} trips")
print("First 10 segments:")
for start_date, start_station, end_station, duration in path_segments[:10]:
    print(f"{start_date}: {start_station} -> {end_station} ({duration} sec)")

unique_stations = (
    max_bike_path
    .flatMap(lambda trip: [trip["start_station_name"], trip["end_station_name"]])
    .distinct()
    .count()
)

print(f"Unique stations visited by bike {bike_id}: {unique_stations}")


Bike 466 path length: 846 trips
First 10 segments:
2013-08-29 18:56: Powell at Post (Union Square) -> Yerba Buena Center of the Arts (3rd @ Howard) (826 sec)
2013-08-29 19:47: Yerba Buena Center of the Arts (3rd @ Howard) -> Powell at Post (Union Square) (1499 sec)
2013-08-31 08:25: Powell at Post (Union Square) -> Powell Street BART (278 sec)
2013-09-01 09:29: Powell Street BART -> Powell Street BART (27647 sec)
2013-09-01 19:38: Powell Street BART -> South Van Ness at Market (478 sec)
2013-09-01 21:53: South Van Ness at Market -> Market at 4th (656 sec)
2013-09-02 12:36: Market at 4th -> Harry Bridges Plaza (Ferry Building) (827 sec)
2013-09-02 13:05: Harry Bridges Plaza (Ferry Building) -> Steuart at Market (22014 sec)
2013-09-02 19:31: Steuart at Market -> Market at 4th (523 sec)
2013-09-03 09:55: Market at 4th -> Market at 4th (23438 sec)
Unique stations visited by bike 466: 37


### Задание 4. Найти количество велосипедов в системе


In [7]:
bike_count = trips.map(lambda trip: trip["bike_id"]).distinct().count()
print(f"Bikes in system: {bike_count}")


Bikes in system: 700


### Задание 5. Найти пользователей, потративших на поездки более 3 часов


In [8]:
three_hours_sec = 3 * 3600

user_total_time = (
    trips
    .filter(lambda trip: trip["zip_code"] != "")
    .map(lambda trip: (trip["zip_code"], trip["duration"]))
    .reduceByKey(lambda left, right: left + right)
)

heavy_users = (
    user_total_time
    .filter(lambda item: item[1] > three_hours_sec)
    .sortBy(lambda item: item[1], ascending=False)
    .cache()
)

print(f"Users above 3 hours: {heavy_users.count()}")
print("Top 10 users by total duration:")
for zip_code, duration in heavy_users.take(10):
    print(f"{zip_code}: {duration} sec ({duration / 3600:.2f} hours)")


Users above 3 hours: 2732
Top 10 users by total duration:
94107: 26357318 sec (7321.48 hours)
nil: 25017541 sec (6949.32 hours)
94105: 17030133 sec (4730.59 hours)
94133: 13034793 sec (3620.78 hours)
94102: 12388496 sec (3441.25 hours)
94103: 10833079 sec (3009.19 hours)
94111: 8927655 sec (2479.90 hours)
94109: 8435951 sec (2343.32 hours)
95112: 7798992 sec (2166.39 hours)
94040: 5575933 sec (1548.87 hours)


В данных нет отдельного `user_id`, поэтому в качестве идентификатора пользователя используется поле `zip_code`.
